# 04 — Task 2.4: Rules-Based Fraud Flagging

**Target Table:** `bfsi.silver_layer.fraud_alerts`

**Objective:** Apply deterministic fraud rules to `transaction_features` and flag suspicious transactions.

| Rule ID | Rule Name | Condition |
|---------|-----------|----------|
| R1 | `VELOCITY_BREACH` | `txn_count_1h > 10` OR `amount_sum_1h > 500,000` |
| R2 | `HIGH_AMOUNT` | `is_international AND txn_amount_inr > 100,000` |
| R3 | `IMPOSSIBLE_TRAVEL` | `distinct_countries_24h > 3` |
| R4 | `BLACKLISTED_MERCHANT` | `merchant.is_blacklisted = TRUE` |
| R5 | `AMOUNT_ANOMALY` | `txn_amount_inr > 10 × customer 30d average` |

**Output:** `fraud_rule_triggered` — array of all fired rule names per transaction.

> Reads from `silver.transaction_features` (Task 2.3) + `bronze.merchants` (for blacklist).

---
## Cell 1 — Configuration & Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import ArrayType, StringType
from datetime import datetime

# ── Catalog / Schema ───────────────────────────────────────────────────
CATALOG       = "bfsi"
BRONZE_SCHEMA = "bronze_layer"
SILVER_SCHEMA = "silver_layer"

# ── Source tables ──────────────────────────────────────────────────────
FEATURES_TABLE  = f"{CATALOG}.{SILVER_SCHEMA}.transaction_features"
MERCHANTS_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.merchants"           # Blacklist lookup

# ── Target table ──────────────────────────────────────────────────────
FRAUD_ALERTS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.fraud_alerts"

# ── Rule version (for auditability) ────────────────────────────────────
RULE_VERSION = "v1.0-task2.4"

# ── Rule thresholds ────────────────────────────────────────────────────
VELOCITY_TXN_COUNT_THRESHOLD   = 10        # max txns in 1 hour
VELOCITY_AMOUNT_THRESHOLD      = 500_000   # max INR spend in 1 hour
INTL_HIGH_AMOUNT_THRESHOLD     = 100_000   # INR for international txns
IMPOSSIBLE_TRAVEL_THRESHOLD    = 3         # distinct countries in 24h
AMOUNT_ANOMALY_MULTIPLIER      = 10        # 10× the 30-day average

# ── Incremental: get watermark from target table ───────────────────────
try:
    _max_ts = (
        spark.read.table(FRAUD_ALERTS_TABLE)
        .agg(F.max("_silver_load_ts").alias("max_ts"))
        .collect()[0]["max_ts"]
    )
    LAST_LOAD_TS = _max_ts if _max_ts is not None else datetime(1900, 1, 1)
except Exception:
    LAST_LOAD_TS = datetime(1900, 1, 1)

print(f"Task 2.4: Rules-Based Fraud Flagging (INCREMENTAL)")
print(f"Rule Version : {RULE_VERSION}")
print(f"Watermark    : {LAST_LOAD_TS}")
print(f"Timestamp    : {datetime.now().isoformat()}")
print(f"\nThresholds:")
print(f"  Velocity    : >{VELOCITY_TXN_COUNT_THRESHOLD} txns/hr OR >₹{VELOCITY_AMOUNT_THRESHOLD:,}/hr")
print(f"  Intl amount : >₹{INTL_HIGH_AMOUNT_THRESHOLD:,} on international txn")
print(f"  Impossible  : >{IMPOSSIBLE_TRAVEL_THRESHOLD} countries in 24h")
print(f"  Anomaly     : >{AMOUNT_ANOMALY_MULTIPLIER}× 30-day average")

---
## Cell 2 — Load Source Data

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  LOAD SOURCE DATA (INCREMENTAL)
# ══════════════════════════════════════════════════════════════════════════

# Transaction features (from Task 2.3) — only new records
df_features = (
    spark.read.table(FEATURES_TABLE)
    .filter(F.col("_silver_load_ts") > F.lit(LAST_LOAD_TS))
)
new_count = df_features.count()
print(f"New records to process: {new_count:,} from {FEATURES_TABLE}")

if new_count == 0:
    print("No new data found. Skipping fraud flagging.")
    dbutils.notebook.exit("NO_NEW_DATA")

# Merchants (for blacklist lookup — from Bronze, read-only)
df_merchants = spark.read.table(MERCHANTS_TABLE)
print(f"Loaded {df_merchants.count():,} merchants from {MERCHANTS_TABLE}")

blacklisted_count = df_merchants.filter(F.col("is_blacklisted") == True).count()
print(f"Blacklisted merchants: {blacklisted_count:,}")

---
## Cell 3 — Join Merchant Blacklist

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  JOIN MERCHANT BLACKLIST
#  Left-join transaction_features with merchants to get is_blacklisted
#  UPI/NEFT have NULL merchant_id → is_blacklisted will be NULL (= false)
# ══════════════════════════════════════════════════════════════════════════

df_with_merchant = (
    df_features
    .join(
        df_merchants.select(
            F.col("merchant_id").alias("_m_id"),
            F.col("is_blacklisted")
        ),
        df_features.merchant_id == F.col("_m_id"),
        how="left"
    )
    .drop("_m_id")
    # Fill NULL blacklist status with False
    .withColumn(
        "is_blacklisted",
        F.coalesce(F.col("is_blacklisted"), F.lit(False))
    )
)

print(f"Joined merchant blacklist. Records: {df_with_merchant.count():,}")
print(f"Transactions at blacklisted merchants: {df_with_merchant.filter(F.col('is_blacklisted') == True).count():,}")

---
## Cell 4 — Compute 30-Day Average for Anomaly Rule

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  COMPUTE 30-DAY AVERAGE PER CUSTOMER
#  Used for Rule R5: amount > 10 × 30d average
#  We use a rolling 30-day window per customer
# ══════════════════════════════════════════════════════════════════════════

THIRTY_DAYS_SEC = 30 * 86400

window_30d = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.unix_timestamp("txn_ts"))
    .rangeBetween(-THIRTY_DAYS_SEC, 0)
)

df_with_avg = df_with_merchant.withColumn(
    "_customer_avg_30d",
    F.avg("txn_amount_inr").over(window_30d)
)

print("30-day rolling average computed per customer.")

---
## Cell 5 — Apply Fraud Rules

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  APPLY ALL FRAUD RULES
#  Each rule produces a boolean column; then we combine into an array
# ══════════════════════════════════════════════════════════════════════════

df_rules = (
    df_with_avg

    # ── R1: VELOCITY_BREACH ────────────────────────────────────────────
    # Triggered if: txn_count_1h > 10 OR amount_sum_1h > 500,000
    .withColumn(
        "_r1_velocity",
        F.when(
            (F.col("txn_count_1h") > VELOCITY_TXN_COUNT_THRESHOLD) |
            (F.col("amount_sum_1h") > VELOCITY_AMOUNT_THRESHOLD),
            F.lit(True)
        ).otherwise(F.lit(False))
    )

    # ── R2: HIGH_AMOUNT (international) ────────────────────────────────
    # Triggered if: is_international = TRUE AND txn_amount_inr > 100,000
    .withColumn(
        "_r2_high_amount",
        F.when(
            (F.col("is_international") == True) &
            (F.col("txn_amount_inr") > INTL_HIGH_AMOUNT_THRESHOLD),
            F.lit(True)
        ).otherwise(F.lit(False))
    )

    # ── R3: IMPOSSIBLE_TRAVEL ──────────────────────────────────────────
    # Triggered if: distinct_countries_24h > 3
    .withColumn(
        "_r3_impossible_travel",
        F.when(
            F.col("distinct_countries_24h") > IMPOSSIBLE_TRAVEL_THRESHOLD,
            F.lit(True)
        ).otherwise(F.lit(False))
    )

    # ── R4: BLACKLISTED_MERCHANT ───────────────────────────────────────
    # Triggered if: merchant is_blacklisted = TRUE
    .withColumn(
        "_r4_blacklisted",
        F.when(
            F.col("is_blacklisted") == True,
            F.lit(True)
        ).otherwise(F.lit(False))
    )

    # ── R5: AMOUNT_ANOMALY ─────────────────────────────────────────────
    # Triggered if: txn_amount_inr > 10 × customer 30-day average
    .withColumn(
        "_r5_amount_anomaly",
        F.when(
            (F.col("_customer_avg_30d").isNotNull()) &
            (F.col("_customer_avg_30d") > 0) &
            (F.col("txn_amount_inr") > AMOUNT_ANOMALY_MULTIPLIER * F.col("_customer_avg_30d")),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
)

print("All 5 fraud rules applied.")
for rule, col_name in [
    ("R1: VELOCITY_BREACH", "_r1_velocity"),
    ("R2: HIGH_AMOUNT", "_r2_high_amount"),
    ("R3: IMPOSSIBLE_TRAVEL", "_r3_impossible_travel"),
    ("R4: BLACKLISTED_MERCHANT", "_r4_blacklisted"),
    ("R5: AMOUNT_ANOMALY", "_r5_amount_anomaly"),
]:
    cnt = df_rules.filter(F.col(col_name) == True).count()
    print(f"  {rule}: {cnt:,} triggers")

---
## Cell 6 — Build fraud_rule_triggered Array

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  BUILD fraud_rule_triggered ARRAY
#  Combines all fired rules into a single array column
#  Example output: ['VELOCITY_BREACH', 'HIGH_AMOUNT']
# ══════════════════════════════════════════════════════════════════════════

df_flagged = (
    df_rules
    .withColumn(
        "fraud_rule_triggered",
        F.array_compact(          # Remove nulls from the array
            F.array(
                F.when(F.col("_r1_velocity") == True, F.lit("VELOCITY_BREACH")),
                F.when(F.col("_r2_high_amount") == True, F.lit("HIGH_AMOUNT")),
                F.when(F.col("_r3_impossible_travel") == True, F.lit("IMPOSSIBLE_TRAVEL")),
                F.when(F.col("_r4_blacklisted") == True, F.lit("BLACKLISTED_MERCHANT")),
                F.when(F.col("_r5_amount_anomaly") == True, F.lit("AMOUNT_ANOMALY")),
            )
        )
    )
    # Number of rules triggered per transaction
    .withColumn(
        "rules_fired_count",
        F.size("fraud_rule_triggered")
    )
    # Boolean flag: is this transaction suspicious?
    .withColumn(
        "is_suspicious",
        F.col("rules_fired_count") > 0
    )
)

total = df_flagged.count()
suspicious = df_flagged.filter(F.col("is_suspicious") == True).count()
clean = total - suspicious

print(f"Total transactions   : {total:,}")
print(f"Suspicious (flagged) : {suspicious:,} ({100*suspicious/total:.2f}%)")
print(f"Clean                : {clean:,} ({100*clean/total:.2f}%)")

# Distribution of rules-fired count
print("\nRules fired distribution:")
display(
    df_flagged
    .filter(F.col("is_suspicious") == True)
    .groupBy("rules_fired_count")
    .agg(F.count("*").alias("txn_count"))
    .orderBy("rules_fired_count")
)

---
## Cell 7 — Filter & Write Fraud Alerts

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  WRITE fraud_alerts TO SILVER DELTA TABLE (MERGE / UPSERT)
#  Only suspicious transactions (at least 1 rule triggered)
#  Uses MERGE on txn_id to prevent duplicate alerts on pipeline re-runs
# ══════════════════════════════════════════════════════════════════════════

# ── Define output columns explicitly (avoids schema-drift issues) ────────
_ALERT_COLUMNS = [
    "txn_id", "customer_id", "txn_channel", "txn_amount_inr", "txn_ts",
    "status", "masked_counterparty", "merchant_id", "is_international",
    "masked_instrument", "txn_count_1h", "amount_sum_1h",
    "distinct_merchants_1h", "distinct_countries_24h",
    "time_since_last_txn_seconds", "amount_zscore",
    "fraud_rule_triggered", "rules_fired_count", "is_blacklisted",
    "rule_fired_ts", "rule_version", "_silver_load_ts",
]

df_alerts = (
    df_flagged
    .filter(F.col("is_suspicious") == True)
    .select(
        # ── Core transaction columns ──
        "txn_id",
        "customer_id",
        "txn_channel",
        "txn_amount_inr",
        "txn_ts",
        "status",
        "masked_counterparty",
        "merchant_id",
        "is_international",
        "masked_instrument",
        # ── Fraud features (from Task 2.3) ──
        "txn_count_1h",
        "amount_sum_1h",
        "distinct_merchants_1h",
        "distinct_countries_24h",
        "time_since_last_txn_seconds",
        "amount_zscore",
        # ── Fraud flagging output ──
        "fraud_rule_triggered",
        "rules_fired_count",
        "is_blacklisted",
    )
    # ── Alert metadata ──
    .withColumn("rule_fired_ts", F.current_timestamp())
    .withColumn("rule_version", F.lit(RULE_VERSION))
    .withColumn("_silver_load_ts", F.current_timestamp())
)

new_candidates = df_alerts.count()
before_count = 0

# ── MERGE into Delta — only insert truly new alerts (by txn_id) ──────────
if not spark.catalog.tableExists(FRAUD_ALERTS_TABLE):
    # First run: table does not exist yet — create it
    df_alerts.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(FRAUD_ALERTS_TABLE)
else:
    before_count = spark.read.table(FRAUD_ALERTS_TABLE).count()
    # Register new alerts as a temp view for SQL MERGE
    df_alerts.createOrReplaceTempView("__new_fraud_alerts")

    # Build explicit column lists to avoid schema-drift (_rn etc.)
    _cols_csv   = ", ".join(_ALERT_COLUMNS)
    _src_csv    = ", ".join([f"source.{c}" for c in _ALERT_COLUMNS])

    spark.sql(f"""
        MERGE INTO {FRAUD_ALERTS_TABLE} AS target
        USING __new_fraud_alerts AS source
        ON target.txn_id = source.txn_id
        WHEN NOT MATCHED THEN INSERT ({_cols_csv})
             VALUES ({_src_csv})
    """)

after_count = spark.read.table(FRAUD_ALERTS_TABLE).count()
actually_inserted = after_count - before_count

print(f"\n Silver table updated: {FRAUD_ALERTS_TABLE}")
print(f"Candidates evaluated : {new_candidates:,}")
print(f"New alerts inserted  : {actually_inserted:,}  (duplicates skipped)")
print(f"Total fraud alerts   : {after_count:,}")

---
## Cell 8 — Verification & Alert Summary

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  TASK 2.4 — VERIFICATION & ALERT SUMMARY
# ══════════════════════════════════════════════════════════════════════════

df_result = spark.read.table(FRAUD_ALERTS_TABLE)

print("=" * 70)
print("  TASK 2.4: RULES-BASED FRAUD FLAGGING — COMPLETE")
print("=" * 70)

# ── Per-rule trigger counts ──
print("\n Rule Trigger Summary:")
rule_names = [
    "VELOCITY_BREACH",
    "HIGH_AMOUNT",
    "IMPOSSIBLE_TRAVEL",
    "BLACKLISTED_MERCHANT",
    "AMOUNT_ANOMALY",
]

print(f"\n{'Rule':<30} {'Triggers':>10}")
print("-" * 42)
for rule in rule_names:
    cnt = df_result.filter(
        F.array_contains(F.col("fraud_rule_triggered"), rule)
    ).count()
    print(f"{rule:<30} {cnt:>10,}")

# ── Per-channel breakdown ──
print("\n Alerts by Channel:")
display(
    df_result
    .groupBy("txn_channel")
    .agg(
        F.count("*").alias("alert_count"),
        F.round(F.sum("txn_amount_inr"), 2).alias("total_flagged_amount"),
        F.round(F.avg("rules_fired_count"), 2).alias("avg_rules_per_alert"),
    )
    .orderBy("txn_channel")
)

# ── Multi-rule triggers ──
multi_rule = df_result.filter(F.col("rules_fired_count") > 1).count()
print(f"\n Multi-rule triggers (>1 rule): {multi_rule:,}")

# ── Schema validation ──
expected = [
    "txn_id", "customer_id", "txn_channel", "txn_amount_inr",
    "fraud_rule_triggered", "rules_fired_count",
    "rule_fired_ts", "rule_version"
]
missing = [c for c in expected if c not in df_result.columns]
print(f" Schema: {'ALL columns present' if not missing else f'MISSING: {missing}'}")

print(f"\n Rule Version: {RULE_VERSION}")
print(f" Total alerts: {df_result.count():,}")
print(f" Timestamp: {datetime.now().isoformat()}")
print("=" * 70)

In [0]:
# ── Sample: Highest-risk transactions (most rules triggered) ───────────
print("Top 10 highest-risk alerts (most rules fired):")
display(
    df_result
    .select(
        "txn_id", "customer_id", "txn_channel",
        "txn_amount_inr", "status",
        "fraud_rule_triggered", "rules_fired_count",
        "txn_count_1h", "amount_sum_1h",
        "distinct_countries_24h", "amount_zscore",
        "is_blacklisted", "rule_version",
    )
    .orderBy(F.col("rules_fired_count").desc(), F.col("txn_amount_inr").desc())
    .limit(10)
)

---

### Fraud Rules Reference

| Rule | Name | Condition | Risk Level |
|------|------|-----------|------------|
| R1 | `VELOCITY_BREACH` | `txn_count_1h > 10` OR `amount_sum_1h > ₹5,00,000` | HIGH |
| R2 | `HIGH_AMOUNT` | International + amount > ₹1,00,000 | MEDIUM |
| R3 | `IMPOSSIBLE_TRAVEL` | > 3 countries in 24 hours | CRITICAL |
| R4 | `BLACKLISTED_MERCHANT` | Merchant on blacklist | HIGH |
| R5 | `AMOUNT_ANOMALY` | Amount > 10× customer's 30-day average | HIGH |

**Multi-rule escalation:** Transactions triggering 2+ rules should be escalated to the fraud investigation team.

> **Next:** Gold Layer — SAR reports, risk scoring dashboards